In [ ]:
# Run once in a fresh environment if required:
# %pip install torch torchvision tensorflow scikit-learn matplotlib seaborn grad-cam
import random, time, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using", DEVICE)
except ImportError:
    print("PyTorch is not installed yet. Run the installation line above.")

In [ ]:
# Question 1. Normalization dynamics and gradient flow
# Brief: Compute sizes through the stated CNN, then evaluate BatchNorm, a residual addition, clipping, and min-max scaling.

In [ ]:
def conv_size(n, k, stride=1, padding=0):
    return (n + 2 * padding - k) // stride + 1

h = w = 64
h = w = conv_size(h, 7, 2, 3); print("Conv1:", (16, h, w))       # 16 x 32 x 32
h = w = conv_size(h, 3, 2, 1); print("MaxPool:", (16, h, w))    # 16 x 16 x 16
print("Residual block:", (16, h, w), "(same padding/stride)")
print("Global average pool:", (16, 1, 1), "; FC input: 16")

x, mu, var, gamma, beta = 20, 12, 16, 0.5, -1
bn_y = gamma * (x - mu) / math.sqrt(var) + beta
print(f"BatchNorm output = {bn_y:.2f}")                         # 0
F = np.array([2, -1, .5]); identity = np.array([1, 3, -.5])
print("Residual output F(x)+x =", F + identity)                 # [3,2,0]
g = np.array([15., 20.]); c = 5
g_clip = g * min(1, c / np.linalg.norm(g))
print("Clipped-gradient expression: g * 5/25; norm:", np.linalg.norm(g_clip))
print("Min-max value:", (20 - (-4)) / (28 - (-4)))             # .75

In [ ]:
# Question 2. Transfer learning and normalization
# Brief: Compare [0,1] pixel scaling against CIFAR-10 z-score scaling for a frozen ResNet-18 head, its last-block fine-tune, and a small VGG-inspired CNN. The cells record loss, validation accuracy, final-layer gradient norm, and test accuracy.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
def cifar_loaders(zscore, batch_size=128):
    transform = transforms.Compose([transforms.ToTensor()] + ([transforms.Normalize(CIFAR_MEAN, CIFAR_STD)] if zscore else []))
    train = datasets.CIFAR10("data", train=True, download=True, transform=transform)
    test = datasets.CIFAR10("data", train=False, download=True, transform=transform)
    train_set, val_set = torch.utils.data.random_split(train, [45000, 5000], generator=torch.Generator().manual_seed(SEED))
    return (DataLoader(train_set, batch_size, shuffle=True, num_workers=2),
            DataLoader(val_set, batch_size, num_workers=2), DataLoader(test, batch_size, num_workers=2))

class SmallVGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                                      nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.Conv2d(64,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(),nn.Linear(64*8*8,128),nn.ReLU(),nn.Dropout(.25),nn.Linear(128,10))
    def forward(self,x): return self.classifier(self.features(x))

def accuracy(model, loader):
    model.eval(); hits = total = 0
    with torch.no_grad():
        for x,y in loader:
            pred = model(x.to(DEVICE)).argmax(1); hits += (pred == y.to(DEVICE)).sum().item(); total += y.numel()
    return hits / total

def train_epochs(model, train_loader, val_loader, epochs, lr=1e-3):
    opt, loss_fn = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=lr), nn.CrossEntropyLoss()
    history = {"loss": [], "val_acc": [], "grad_norm": []}
    for epoch in range(epochs):
        model.train(); running = 0
        for x,y in train_loader:
            opt.zero_grad(); out = model(x.to(DEVICE)); loss = loss_fn(out, y.to(DEVICE)); loss.backward()
            final_weight = next(p for n,p in model.named_parameters() if "classifier" in n and p.grad is not None)
            history["grad_norm"].append(final_weight.grad.norm().item())
            opt.step(); running += loss.item() * y.size(0)
        history["loss"].append(running / len(train_loader.dataset)); history["val_acc"].append(accuracy(model, val_loader))
        print(f"epoch {epoch+1}: loss={history['loss'][-1]:.3f}, val_acc={history['val_acc'][-1]:.3f}")
    return history

def run_transfer(zscore=True):
    train_loader, val_loader, test_loader = cifar_loaders(zscore)
    net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in net.parameters(): p.requires_grad = False
    net.fc = nn.Linear(net.fc.in_features, 10); net = net.to(DEVICE)
    head_history = train_epochs(net, train_loader, val_loader, 5)
    for p in net.layer4.parameters(): p.requires_grad = True
    fine_history = train_epochs(net, train_loader, val_loader, 10, lr=1e-4)
    return net, {k: head_history[k] + fine_history[k] for k in head_history}, accuracy(net, test_loader)

# Uncomment for the four requested experiments. Each has genuine saved curves.
# resnet_mm, hist_resnet_mm, test_mm = run_transfer(False)
# resnet_z, hist_resnet_z, test_z = run_transfer(True)
# For SmallVGG, call train_epochs(SmallVGG().to(DEVICE), *cifar_loaders(True)[:2], epochs=15).

In [ ]:
def plot_history(histories, metric, ylabel):
    plt.figure(figsize=(7,4))
    for label, h in histories.items(): plt.plot(h[metric], label=label)
    plt.xlabel("Epoch" if metric != "grad_norm" else "Training step"); plt.ylabel(ylabel); plt.legend(); plt.grid(alpha=.25); plt.show()

# After the runs: plot_history({"ResNet min-max":hist_resnet_mm, "ResNet z-score":hist_resnet_z}, "loss", "Training loss")
# plot_history({"ResNet min-max":hist_resnet_mm, "ResNet z-score":hist_resnet_z}, "val_acc", "Validation accuracy")
# plot_history({"ResNet min-max":hist_resnet_mm, "ResNet z-score":hist_resnet_z}, "grad_norm", "Final-layer gradient norm")

**Discussion.** Pretrained filters begin with useful edge, texture, and shape detectors, so the classifier usually converges earlier and is less able to memorize a small dataset: this is a practical form of implicit regularization. Z-score normalization matches the distribution normally used by ImageNet weights and often yields more consistently scaled gradients than only [0,1] scaling. Residual paths provide an identity route for both activations and gradients; a plain CNN must learn every transformation and is therefore harder to optimize as it deepens.

In [ ]:
# Question 3. BatchNorm, LayerNorm, and gradient clipping
# Brief: State the formulas, dimensions, optimization effects, and why clipping controls unstable updates.

For a batch X with shape B x d, BatchNorm works feature by feature. For feature j:

`mean_j = (1 / B) * sum_i X[i, j]`

`var_j = (1 / B) * sum_i (X[i, j] - mean_j)^2`

`Y[i, j] = gamma_j * (X[i, j] - mean_j) / sqrt(var_j + epsilon) + beta_j`

The mean and variance are taken across the batch dimension. In a convolutional layer, they are taken over batch and spatial positions for each channel.

LayerNorm normalizes each sample independently across its d features:

`mean_i = (1 / d) * sum_j X[i, j]`

`var_i = (1 / d) * sum_j (X[i, j] - mean_i)^2`

Small batches give BatchNorm noisy mean/variance estimates and create a train/evaluation mismatch. LayerNorm has no such dependency because its statistics come from one sample. Transformers commonly use variable sequence lengths and small effective batches, so LayerNorm is more reliable and does not mix information between examples.

Normalization keeps activation scales comparable across layers and usually gives better-conditioned optimization, allowing stable larger updates. Internal covariate shift is a useful intuition, but it is not the whole explanation: normalization also changes the parameterization and smooths optimization.

Norm clipping is: `g_clipped = g * min(1, c / (||g||_2 + epsilon))`. If `||g||_2 <= c`, nothing changes. Otherwise the clipped gradient norm is exactly c, so it is never larger than c. This prevents a single exploding gradient from causing an unstable update in deep networks and RNNs; it does not fix vanishing gradients.

In [ ]:
# Question 4. Oxford-IIIT Pet segmentation with U-Net
# Brief: Implement the requested 3-32-64-128 encoder, 256 bottleneck, decoder with concatenative skips, then train an ablation without skips and compare losses, IoU, and prediction maps.

In [ ]:
from PIL import Image
class OxfordPetsSegmentation(torch.utils.data.Dataset):
    def __init__(self, root="data", split="train", image_size=128):
        base = datasets.OxfordIIITPet(root=root, download=True, target_types="segmentation")
        cut = int(.8 * len(base._images)); ids = range(0,cut) if split == "train" else range(cut,len(base._images))
        self.images, self.masks = [base._images[i] for i in ids], [base._segs[i] for i in ids]
        self.image_tf = transforms.Compose([transforms.Resize((image_size,image_size)),transforms.ToTensor()])
        self.mask_tf = transforms.Compose([transforms.Resize((image_size,image_size), interpolation=transforms.InterpolationMode.NEAREST),transforms.PILToTensor()])
    def __len__(self): return len(self.images)
    def __getitem__(self,i): return self.image_tf(Image.open(self.images[i]).convert("RGB")), (self.mask_tf(Image.open(self.masks[i])).squeeze(0)-1).long()

class DoubleConv(nn.Module):
    def __init__(self, a,b): super().__init__(); self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.ReLU())
    def forward(self,x): return self.net(x)
class UNet(nn.Module):
    def __init__(self, skips=True):
        super().__init__(); self.skips=skips; self.e1,self.e2,self.e3=DoubleConv(3,32),DoubleConv(32,64),DoubleConv(64,128); self.pool=nn.MaxPool2d(2); self.b=DoubleConv(128,256)
        self.u3,self.d3=nn.ConvTranspose2d(256,128,2,2),DoubleConv(256 if skips else 128,128)
        self.u2,self.d2=nn.ConvTranspose2d(128,64,2,2),DoubleConv(128 if skips else 64,64)
        self.u1,self.d1=nn.ConvTranspose2d(64,32,2,2),DoubleConv(64 if skips else 32,32); self.out=nn.Conv2d(32,3,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b)); x=self.b(self.pool(c))
        x=self.u3(x); x=self.d3(torch.cat([x,c],1) if self.skips else x); x=self.u2(x); x=self.d2(torch.cat([x,b],1) if self.skips else x); x=self.u1(x); x=self.d1(torch.cat([x,a],1) if self.skips else x); return self.out(x)

def mean_iou(logits, mask, classes=3):
    pred=logits.argmax(1); vals=[]
    for cls in range(classes):
        inter=((pred==cls)&(mask==cls)).sum().item(); union=((pred==cls)|(mask==cls)).sum().item()
        vals.append(inter/union if union else float("nan"))
    return np.nanmean(vals)

# Train both variants for 25--30 epochs using the generic loop below.
def train_segmenter(model, train_loader, val_loader, epochs=25):
    opt=torch.optim.Adam(model.parameters(),lr=1e-3); criterion=nn.CrossEntropyLoss(); hist={"train":[],"val":[],"iou":[]}
    for e in range(epochs):
        model.train(); total=0
        for x,y in train_loader:
            opt.zero_grad(); loss=criterion(model(x.to(DEVICE)),y.to(DEVICE)); loss.backward(); opt.step(); total+=loss.item()
        model.eval(); vl=[]; ious=[]
        with torch.no_grad():
            for x,y in val_loader:
                z=model(x.to(DEVICE)); vl.append(criterion(z,y.to(DEVICE)).item()); ious.append(mean_iou(z.cpu(),y))
        hist["train"].append(total/len(train_loader)); hist["val"].append(np.mean(vl)); hist["iou"].append(np.mean(ious)); print(e+1,hist["val"][-1],hist["iou"][-1])
    return hist

**Analysis.** Concatenative skips restore high-resolution boundary and texture information that pooling discards, so removing them typically increases blurred boundaries, small-part omissions, and class leakage around outlines. A U-Net skip joins encoder features to a decoder representation (channel concatenation); a ResNet skip adds same-shaped features to make residual optimization easy. They serve complementary purposes.

In [ ]:
# Question 5: Effect of stride and padding in convolution

In [ ]:
# Question 6: Residual connections and optimization
# Brief: Calculate convolution geometry and explain why identity paths train deeply.

In [ ]:
# Q5: floor((64 + 2*2 - 5)/2)+1 = 32, so output is 32 x 32 x 32.
# Parameters = 32 * (3*5*5 + 1 bias) = 2,432.
print("Q5 output: 32 x 32 x 32; parameters:", 32*(3*5*5+1))
print("Q6: if F(x)=0, y=x. The identity derivative adds a direct gradient path, so blocks can learn small residual corrections rather than an entire mapping.")

In [ ]:
# Question 7. California Housing preprocessing
# Brief: Train the exact 64-64-32-16 MLP for 100 epochs under raw, min-max, and z-score preprocessing, fitting scalers only on the training partition.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error
class HousingMLP(nn.Module):
    def __init__(self, d): super().__init__(); self.net=nn.Sequential(nn.Linear(d,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
    def forward(self,x): return self.net(x).squeeze(1)
def housing_run(kind, epochs=100):
    X,y=fetch_california_housing().data,fetch_california_housing().target
    Xtr,Xva,ytr,yva=train_test_split(X,y,test_size=.2,random_state=42)
    scaler={"minmax":MinMaxScaler(),"zscore":StandardScaler()}.get(kind)
    if scaler: Xtr,Xva=scaler.fit_transform(Xtr),scaler.transform(Xva)  # no leakage
    xtr,xva=torch.tensor(Xtr,dtype=torch.float32),torch.tensor(Xva,dtype=torch.float32); ytr,yva=torch.tensor(ytr,dtype=torch.float32),torch.tensor(yva,dtype=torch.float32)
    m=HousingMLP(Xtr.shape[1]); opt=torch.optim.Adam(m.parameters(),lr=.001); loss_fn=nn.MSELoss(); curve=[]
    for _ in range(epochs): opt.zero_grad(); loss=loss_fn(m(xtr),ytr); loss.backward(); opt.step(); curve.append(loss_fn(m(xva),yva).item())
    return curve
# curves={name:housing_run(name) for name in ["minmax","zscore","raw"]}
# for name,c in curves.items(): plt.plot(c,label=name)
# plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Validation MSE"); plt.show()

In [ ]:
# Question 8. Grid versus random search on CIFAR-100
# Brief: Execute exactly nine grid runs and nine seeded random runs; rank the top three by validation accuracy and retrain/evaluate the global winner.

In [ ]:
# Reuse a compact implementation so all 18 trials have identical data and 15-epoch budgets.
def cifar100_model(dropout):
    layers=[]; channels=[3,32,64,128]
    for a,b in zip(channels[:-1],channels[1:]): layers += [nn.Conv2d(a,b,3,padding=1),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.ReLU(),nn.MaxPool2d(2)]
    return nn.Sequential(*layers,nn.Flatten(),nn.Linear(128*4*4,512),nn.ReLU(),nn.Dropout(dropout),nn.Linear(512,100)).to(DEVICE)
def search_trial(lr,dropout,train_loader,val_loader,epochs=15):
    model=cifar100_model(dropout); opt=torch.optim.Adam(model.parameters(),lr=lr); loss_fn=nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for x,y in train_loader: opt.zero_grad(); loss=loss_fn(model(x.to(DEVICE)),y.to(DEVICE)); loss.backward(); opt.step()
    return {"lr":lr,"dropout":dropout,"loss":loss.item(),"val_acc":accuracy(model,val_loader),"model":model}
grid=[(lr,do) for lr in [.0005,.001,.005] for do in [.2,.4,.6]]
rng=np.random.default_rng(SEED); random_trials=list(zip(10**rng.uniform(-4,-2,9),rng.uniform(.1,.6,9)))
# grid_results=[search_trial(lr,d,train100,val100) for lr,d in grid]
# random_results=[search_trial(lr,d,train100,val100) for lr,d in random_trials]
# import pandas as pd
# pd.DataFrame(grid_results).drop(columns="model").nlargest(3,"val_acc"), pd.DataFrame(random_results).drop(columns="model").nlargest(3,"val_acc")

In [ ]:
# Question 9. Large versus small kernels
# Brief: Compare output shapes and parameter economy.

In [ ]:
design_a=64*(3*7*7+1)                         # 9472
design_b=32*(3*3*3+1)+64*(32*3*3+1)          # 19392
print("A: 64x32x32, params",design_a)
print("B: 64x32x32 (after conv-pool-conv), params",design_b, "ratio",design_b/design_a)

In [ ]:
# Question 10. CIFAR-10 feature maps
# Brief: Train the specified CNN, display six feature maps after each convolution, and inspect pooling's spatial compression.

In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.c1=nn.Conv2d(3,16,3,padding=1); self.c2=nn.Conv2d(16,32,3,padding=1); self.c3=nn.Conv2d(32,64,3,padding=1); self.pool=nn.MaxPool2d(2); self.fc1=nn.Linear(64*8*8,128); self.fc2=nn.Linear(128,10)
    def forward(self,x,features=False):
        a=torch.relu(self.c1(x)); p1=self.pool(a); b=torch.relu(self.c2(p1)); p2=self.pool(b); c=torch.relu(self.c3(p2)); out=self.fc2(torch.relu(self.fc1(c.flatten(1))))
        return (out,(a,b,c,p1,p2)) if features else out
def show_maps(feature, title):
    fig,axes=plt.subplots(1,6,figsize=(12,2));
    for i,ax in enumerate(axes): ax.imshow(feature[0,i].detach().cpu(),cmap="viridis"); ax.axis("off")
    fig.suptitle(title); plt.show()
# Train FeatureCNN for 50 epochs with train_epochs adapted to its final layers, then:
# logits,(c1,c2,c3,p1,p2)=feature_net(sample_image.to(DEVICE),features=True)
# show_maps(c1,"Conv1"); show_maps(c2,"Conv2"); show_maps(c3,"Conv3")

**Observations to validate against the produced plots.** First-layer maps normally respond to edges and colour contrast; deeper maps are sparser and combine them into textures/parts. Pooling halves resolution (32→16→8) while retaining strong local responses. Compare training and test curves: a widening gap indicates overfitting; the stabilization epoch is where test accuracy ceases to improve materially.

In [ ]:
# Question 11: Weight initialization in CNNs with ReLU

In [ ]:
# Question 12: Hyperparameter optimization strategies
# Brief: Calculate Kaiming values and distinguish model fitting from hyperparameter search.

In [ ]:
layers=[("Conv1",3*3*3,64*3*3),("Conv2",64*3*3,128*3*3),("Conv3",128*3*3,256*3*3),("FC1",4096,1024),("FC2",1024,10)]
for name,fan_in,fan_out in layers:
    sigma=math.sqrt(2/fan_in); bound=math.sqrt(6/fan_in)
    print(f"{name}: fan_in={fan_in}, fan_out={fan_out}, sigma={sigma:.6f}, uniform bound=±{bound:.6f}")

For ReLU networks, Kaiming normal initializes each weight from a normal distribution with mean 0 and standard deviation `sqrt(2 / fan_in)`. For a convolution, `fan_in = input_channels * kernel_height * kernel_width`.

Kaiming uniform samples each weight between `-sqrt(6 / fan_in)` and `+sqrt(6 / fan_in)`. It has the same target variance, `2 / fan_in`, as the normal version. The code cell above prints fan-in, fan-out, standard deviation, and uniform bounds for every requested layer.

Bayesian optimization builds a surrogate of validation performance and selects informative trials. Grid search tests a fixed lattice; random search samples blindly. Bayesian optimization is usually more sample-efficient when a run is expensive. It cannot replace gradient descent: it chooses a small number of external settings, rather than optimizing millions of differentiable model weights. Gradient descent is usually not directly available for hyperparameters because validation performance after training is expensive, noisy, sometimes discrete, and does not normally provide a usable derivative with respect to those settings.

In [ ]:
# Question 13. Multi-class segmentation labels
# Brief: Extend binary pet/background labels to multiple semantic classes.

A single binary mask is not enough because it says only foreground versus background and loses object identity. Use one integer class label per pixel: `Y` has shape `(H, W)` and each entry is an integer from 0 to C - 1. A one-hot alternative has shape `(H, W, C)`.

The model should emit logits of shape `(batch_size, C, H, W)`. Use multiclass cross-entropy: for each pixel, take the negative log probability assigned to its true class and average over pixels. When background dominates, class-weighted cross-entropy, Dice loss, or focal loss gives small classes more influence during training.

In [ ]:
# Question 14. Gradient clipping during ResNet-18 CIFAR-100 fine-tuning
# Brief: Fine-tune the last residual block for four clipping thresholds, record before/after norms and retained signal, then compare validation performance.

In [ ]:
def grad_norm(model): return math.sqrt(sum((p.grad.detach().norm(2).item()**2) for p in model.parameters() if p.requires_grad and p.grad is not None))
def clipping_run(threshold, train_loader, val_loader, epochs=10):
    m=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in m.parameters(): p.requires_grad=False
    for p in m.layer4.parameters(): p.requires_grad=True
    m.fc=nn.Linear(m.fc.in_features,100); m=m.to(DEVICE); opt=torch.optim.Adam(filter(lambda p:p.requires_grad,m.parameters()),lr=1e-4); loss_fn=nn.CrossEntropyLoss(); h={"before":[],"after":[],"retained":[],"val_acc":[]}
    for _ in range(epochs):
        m.train()
        for x,y in train_loader:
            opt.zero_grad(); loss_fn(m(x.to(DEVICE)),y.to(DEVICE)).backward(); before=grad_norm(m); torch.nn.utils.clip_grad_norm_(filter(lambda p:p.requires_grad,m.parameters()),threshold); after=grad_norm(m); opt.step(); h["before"].append(before); h["after"].append(after); h["retained"].append(after/(before+1e-12))
        h["val_acc"].append(accuracy(m,val_loader))
    return h
# clipping={c:clipping_run(c,train100,val100) for c in [5.,1.,.5,.1]}

The total gradient norm is `||g||_2 = sqrt(sum_i ||g_i||_2^2)`, where `g_i` is the gradient tensor for trainable parameter i. It stays unchanged when it is at or below the threshold c; otherwise clipping makes it exactly c.

The retained-signal ratio is `r = ||g_clipped||_2 / ||g||_2`. A value near 1 means little clipping; a value near 0 means severe suppression. Threshold 5.0 generally retains the most signal and 0.1 retains the least. If 0.1 produces very smooth norm curves but worse validation accuracy than 1.0, the updates are probably too small for the pretrained features to adapt.

In [ ]:
# Question 15. Multi-channel convolution
# Brief: Implement filter-bank convolution manually and compare geometry, operations, and retained spatial detail.

In [ ]:
def conv2d_multi(X,W,stride=1,padding=0):
    # X: C,H,W; W: F,C,K,K. Bias omitted because none was specified.
    C,H,Wi=X.shape; F,Cw,K,_=W.shape; assert C==Cw
    Xp=np.pad(X,((0,0),(padding,padding),(padding,padding)))
    out_h=(H+2*padding-K)//stride+1; out_w=(Wi+2*padding-K)//stride+1; out=np.zeros((F,out_h,out_w))
    for f in range(F):
        for r in range(out_h):
            for col in range(out_w): out[f,r,col]=np.sum(Xp[:,r*stride:r*stride+K,col*stride:col*stride+K]*W[f])
    return out
X=np.random.default_rng(SEED).normal(size=(3,32,32)); W=np.random.default_rng(SEED+1).normal(size=(8,3,3,3))
for s,p in [(1,1),(2,1),(1,0)]:
    y=conv2d_multi(X,W,s,p); ops=np.prod(y.shape)*3*3*3
    print(f"stride={s}, padding={p}: shape={y.shape}, multiply-accumulates={ops:,}")